# Implementation of the equations that make up the GSW functions for calculating the seawater surface density. A summary of the necessary steps are given in the cell below. 

# Mathematical equations

The **main equation** used is: 

$$
\hat{v} (S_A \Theta, p ) = v_u \sum_{i,j,k} v_{ijk} s^i \tau^j \pi^k 
$$

Where the values associated with i,j,k are given in the TEOS10 manual in Appendix K. 

$$
s = \sqrt{\frac{S_A + 24 gkg^-1}{S_{A_u}}}, \quad S_{A_u} = \frac{40 \times 35.16504 g kg^{-1}}{35}
$$

$$
\tau = \frac{\Theta}{\Theta_u}, \quad \Theta_u = 40^{\circ} C  
$$

$$
\pi = \frac{p}{p_u}, \quad P_u = 10^4 dbar
$$

$$
v_u = 1m^{3} kg^{-1}, \quad 
$$

However, p = 0 in this function and therefore $\pi \rightarrow 0$ when $k > 0$. Thats why the GSW function only needs salinity and temperature, not pressure for calculating the surface density: 

$$\hat{v} (S_A, \Theta, p = 0)$$


**Furthermore**, we also need to calculate the conserved temperature to use in the main equation. The following related equations are dedicated to this process.

For calculating the conserved temperature, we need to calculate both the entropy and a potential entalpi! The following steps are therefore needed in the calculations:

1. Calculation of the entropy (equation 2.10.1 from the TEOS manual): 

$$
\eta = \eta(S_a, t, p) = -g_T = - \frac{\partial g_T}{\partial T}
$$

In TEOS - empirical values are already used to create polynomial functions of $\eta$ - and I will rather use these estimated values directly with the aimn to reduce the need for computational calculations. These approximated values are found in appendix B of the TEOS and Copernicus manual! https://os.copernicus.org/articles/19/1719/2023/os-19-1719-2023.pdf

2. An iterativ calculation of potential temperature for the surface - ie. $\theta_0$. This is calculated using the Newton-Raphson iterative technique as illustrated in the equation below, found from TEOS10. It can also be estimated as an integral of the adiabatic lapse rate (Fofonoff, 1962 \& 1985), but I also suspect this would require potentially unneseccary computational power. 

$$
\eta (S_A, \theta, p_r) = \eta (S_A, t, p)
$$

3. Then we have to calculate the potential entalpy, where the reference pressure is always set to be zero because most heat flux activity is near the sea-surface. The reference pressure $p_r = 0$ dbar. This is calculated by using: 

$$
h^0 (S_A, t, p) = h (S_A, \theta, 0) = g(S_A, \theta, 0) - (T_0 + \theta)g_T (S_A, \theta, 03)
$$ 

Shortened to (I think - this is my doing hehe):
$$
h^0 = G - T * g_t 
$$

4. And yuhu now we can finally calculate the conserved temperature! Again with the use of the TEOS equations: 

$$
\Theta (S_A, t, p) = \tilde{\Theta} (S_A, \theta) = \frac{\tilde{h^0}(S_A, \theta)}{c_p^0}

$$

In [ ]:
def Gibbs(tau, salinity):
    #The Gibbs empirical values gathered from Copernicus - for easier calculations of the Gibbs polynomials
    
    """
    Constants
    """
    ETA00 = - 3.7102436569e-01
    ETA10 = 3.0834502223e-04 
    ETA20 = - 3.2916987818e+00
    ETA30 =  7.2818259040e+00 
    ETA40 = - 5.6657256773e+00
    ETA50 =  2.8402903938e+00 
    ETA60 = - 8.9615123138e-01
    ETA70 = 1.0035964794e-01
    ETA80 = 1.8140964105e-03
    ETA01 = 3.0779211774e-02
    ETA11 = 1.5006196848e-03
    ETA21 = 1.2029316021e-01
    ETA31 = 3.7464975805e-01
    ETA41 = - 6.0590428227e-01
    ETA51 = 6.4365865093e-02
    ETA61 = 2.4626795446e-02
    ETA71 = - 1.0335853091e-02
    ETA02 = 2.3045093877e+00
    ETA12 = - 5.4154968624e-03
    ETA22 = - 2.5098282844e+00
    ETA32 = 1.9163697628e-02
    ETA42 = 9.6230320461e-02
    ETA52 = 3.7953034101e-02
    ETA62 = - 5.1206778774e-04

    ETA03 = - 8.4974032876e-01
    ETA13 = - 1.3727475447e-02
    ETA23 = 8.6969911602e-01
    ETA33 = 1.1127539375e-01
    ETA43 = - 8.7616123860e-02
    ETA53 = - 1.6250024449e-02
    ETA04 = 4.1807750439e-01
    ETA14 = 5.1388181100e-02
    ETA24 = - 3.1917000611e-01
    ETA34 = - 4.4999965986e-02
    ETA44 = 3.3822211876e-02
    ETA05 = - 1.9191736060e-01
    ETA15 = - 5.3890029514e-02
    ETA25 = 9.3472917957e-02
    ETA35 = - 4.9779616704e-04
    ETA06 = 6.6066546976e-02
    ETA16 = 2.4144978278e-02
    ETA26 = -1.2850921670e-02
    ETA07 = -1.3678360946e-02
    ETA17 = -4.1337102429e-03
    ETA08 = 1.1180283076e-03

    #Then we calculate the polynomial values
    gibbs_poly_sum = ((((((((ETA08 * tau + ETA17 * salinity + ETA07) * tau
    + (ETA26 * salinity + ETA16) * salinity + ETA06) * tau
    + ((ETA35 * salinity + ETA25) * salinity + ETA15) * salinity
    + ETA05) * tau + (((ETA44 * salinity + ETA34) * salinity
    + ETA24) * salinity + ETA14) * salinity + ETA04) * tau
    + ((((ETA53 * salinity + ETA43) * salinity
    + ETA33) * salinity + ETA23) * salinity + ETA13) * salinity
    + ETA03) * tau + (((((ETA62 * salinity + ETA52) * salinity
    + ETA42) * salinity + ETA32) * salinity
    + ETA22) * salinity + ETA12) * salinity + ETA02) * tau
    + ((((((ETA71 * salinity + ETA61) * salinity + ETA51) * salinity
    + ETA41) * salinity + ETA31) * salinity + ETA21) * salinity + ETA11) * salinity
    + ETA01) * tau + (((((((ETA80 * salinity + ETA70) * salinity
    + ETA60) * salinity + ETA50) * salinity + ETA40) * salinity + ETA30) * salinity
    + ETA20) * salinity + ETA10) * salinity + ETA00)

    return gibbs_poly_sum

In [ ]:
def iterative_potential_temp():
    theta = t
    tolerance = 1e-14 #as stated in potential temperature section 3.1 from IOC et al 2010
    max_iterations = 100 

    for i in range(max_iterations):
        
